<a class="anchor" id='import'>
<font color = '#006400'>
    
# **1. Data Integration** </font>
</a>

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **1.1. Import the needed libraries** </font>

In [5]:
import pandas as pd

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **1.2. Integrate the datasets into the notebook** </font>

In [6]:
#Import u.data dataset
#ratings_path = "../data/ml-100k/u.data"

url_data = "https://files.grouplens.org/datasets/movielens/ml-100k/u.data"
columns = ['userId', 'movieId', 'rating', 'timestamp']

ratings = pd.read_csv(url_data, sep='\t', names=columns)

In [7]:
#Import u.item dataset
#items_path = "../data/ml-100k/u.item"

url_item = "https://files.grouplens.org/datasets/movielens/ml-100k/u.item"

item_columns = [
    'movieId', 'title', 'release_date', 'video_release_date', 'IMDb_URL',
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy',
    'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
    'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

movies = pd.read_csv(url_item, sep='|', names=item_columns, encoding='latin-1')


In [8]:
#Import u.user dataset
#users_path = "../data/ml-100k/u.user"

url_user = "https://files.grouplens.org/datasets/movielens/ml-100k/u.user"

user_columns = ['userId', 'age', 'gender', 'occupation', 'zip_code']

users = pd.read_csv(url_user, sep='|', names=user_columns)

In [9]:
#Import u.genre dataset
#genre_path = "../data/ml-100k/u.genre"

url_genre = "https://files.grouplens.org/datasets/movielens/ml-100k/u.genre"

genre_columns = ['genre', 'genreId']

genres = pd.read_csv(url_genre, sep='|', names=genre_columns, encoding='latin-1')


In [10]:
#Import u.occupation dataset
#occupation_path = "../data/ml-100k/u.occupation"

url_occupation = "https://files.grouplens.org/datasets/movielens/ml-100k/u.occupation"

occupations = pd.read_csv(url_occupation, names=['occupation'], encoding='latin-1')

<a class="anchor" id='import'>
<font color = '#006400'>
    
# **2. Data Access, Exploration and Understanding** </font>
</a>

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.1. Ratings** </font>

In [11]:
def cleaning_ratings_data(df):
    # Remove empty columns
    columns_to_remove = []
    for column in df.columns:
        if len(df[column].value_counts()) == 0:
            columns_to_remove.append(column)
    
    # Remove all identified empty columns
    if columns_to_remove:
        df.drop(columns_to_remove, axis=1, inplace=True)
    
    # Check if ratings are between 1 and 5
    rating_column = 'rating'
    if rating_column in df.columns:
        invalid_ratings = df[(df[rating_column] < 1) | (df[rating_column] > 5)]
        df = df[(df[rating_column] >= 1) & (df[rating_column] <= 5)]
    
    # Remove duplicate user-movie ratings
    df = df.drop_duplicates(subset=['userId', 'movieId'])
    
    # Process timestamps
    df['rating_date'] = pd.to_datetime(df['timestamp'], unit='s')
    df['rating_year'] = df['rating_date'].dt.year
    df['rating_month'] = df['rating_date'].dt.month
    
    return df

ratings = cleaning_ratings_data(ratings)


In [12]:
# Display the processed ratings table and summary
print("PROCESSED RATINGS TABLE")
print("=" * 50)
print(ratings.head())

print("\n" + "=" * 50)
print("PROCESSING SUMMARY")
print("=" * 50)
print(f"Total ratings: {len(ratings)}")
print(f"Columns: {list(ratings.columns)}")
print(f"Rating range: {ratings['rating'].min()} - {ratings['rating'].max()}")
print(f"Date range: {ratings['rating_year'].min()} - {ratings['rating_year'].max()}")
print(f"Unique users: {ratings['userId'].nunique()}")
print(f"Unique movies: {ratings['movieId'].nunique()}")

PROCESSED RATINGS TABLE
   userId  movieId  rating  timestamp         rating_date  rating_year  \
0     196      242       3  881250949 1997-12-04 15:55:49         1997   
1     186      302       3  891717742 1998-04-04 19:22:22         1998   
2      22      377       1  878887116 1997-11-07 07:18:36         1997   
3     244       51       2  880606923 1997-11-27 05:02:03         1997   
4     166      346       1  886397596 1998-02-02 05:33:16         1998   

   rating_month  
0            12  
1             4  
2            11  
3            11  
4             2  

PROCESSING SUMMARY
Total ratings: 100000
Columns: ['userId', 'movieId', 'rating', 'timestamp', 'rating_date', 'rating_year', 'rating_month']
Rating range: 1 - 5
Date range: 1997 - 1998
Unique users: 943
Unique movies: 1682


In [13]:
user_ratings = ratings[ratings['userId'] == 196]
user_ratings

,userId,movieId,rating,timestamp,rating_date,rating_year,rating_month
0,196,242,3,881250949,1997-12-04 15:55:49,1997,12
940,196,393,4,881251863,1997-12-04 16:11:03,1997,12
1133,196,381,4,881251728,1997-12-04 16:08:48,1997,12
1812,196,251,3,881251274,1997-12-04 16:01:14,1997,12
1896,196,655,5,881251793,1997-12-04 16:09:53,1997,12
2374,196,67,5,881252017,1997-12-04 16:13:37,1997,12
6910,196,306,4,881251021,1997-12-04 15:57:01,1997,12
7517,196,238,4,881251820,1997-12-04 16:10:20,1997,12
7842,196,663,5,881251911,1997-12-04 16:11:51,1997,12
10017,196,111,4,881251793,1997-12-04 16:09:53,1997,12


In [14]:
#checking whether user 196 rated the same movie more than once
user_ratings[user_ratings['movieId'].duplicated()]

,userId,movieId,rating,timestamp,rating_date,rating_year,rating_month


In [15]:
ratings.describe()

,userId,movieId,rating,timestamp,rating_date,rating_year,rating_month
count,100000.00000,100000.000000,100000.000000,1.000000e+05,100000,100000.000000,100000.00000
mean,462.48475,425.530130,3.529860,8.835289e+08,1997-12-31 00:40:51.488619904,1997.471010,6.81569
min,1.00000,1.000000,1.000000,8.747247e+08,1997-09-20 03:05:10,1997.000000,1.00000
25%,254.00000,175.000000,3.000000,8.794487e+08,1997-11-13 19:18:29.500000,1997.000000,2.00000
50%,447.00000,322.000000,4.000000,8.828269e+08,1997-12-22 21:42:24,1997.000000,9.00000
75%,682.00000,631.000000,4.000000,8.882600e+08,1998-02-23 18:53:04,1998.000000,11.00000
max,943.00000,1682.000000,5.000000,8.932866e+08,1998-04-22 23:10:38,1998.000000,12.00000
std,266.61442,330.798356,1.125674,5.343856e+06,NaN,0.499161,4.32036


In [16]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   userId        100000 non-null  int64         
 1   movieId       100000 non-null  int64         
 2   rating        100000 non-null  int64         
 3   timestamp     100000 non-null  int64         
 4   rating_date   100000 non-null  datetime64[ns]
 5   rating_year   100000 non-null  int32         
 6   rating_month  100000 non-null  int32         
dtypes: datetime64[ns](1), int32(2), int64(4)
memory usage: 4.6 MB


<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.2. Movies** </font>

In [17]:
movies.head()

,movieId,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [18]:
def cleaning_movies_data(movies):
    # Remove empty columns and irrelevant columns
    columns_to_remove = []
    
    # Remove empty columns (video_release_date, unknown)
    for column in movies.columns:
        if len(movies[column].value_counts()) == 0:
            columns_to_remove.append(column)
    
    # Remove irrelevant columns (IMDb_URL and unknown)
    irrelevant_cols = ['IMDb_URL', 'unknown']
    for col in irrelevant_cols:
        if col in movies.columns and col not in columns_to_remove:
            columns_to_remove.append(col)
    
    # Remove all identified columns
    if columns_to_remove:
        movies.drop(columns_to_remove, axis=1, inplace=True)
    
    # Remove duplicates
    movies = movies.drop_duplicates()
    
    # Process dates
    movies['release_date'] = pd.to_datetime(movies['release_date'])
    movies['release_date_year'] = movies['release_date'].dt.year
    movies['release_date_month'] = movies['release_date'].dt.month
    
    # Remove invalid release years
    invalid_mask = movies['release_date_year'] < 1900
    invalid_count = invalid_mask.sum()
    if invalid_count > 0:
        movies = movies[~invalid_mask]
    
    # Clean titles
    movies['title'] = movies['title'].str.lower()
    movies['title'] = movies['title'].str.replace(r'\s*\(\d{4}\)$', '', regex=True)
    
    return movies

movies = cleaning_movies_data(movies)

In [19]:
# Display the processed movies table and summary
print("PROCESSED MOVIES TABLE")
print("=" * 50)
print(movies.head())

print("\n" + "=" * 50)
print("PROCESSING SUMMARY")
print("=" * 50)
print(f"Total movies: {len(movies)}")
print(f"Columns: {list(movies.columns)}")
print(f"Date range: {movies['release_date_year'].min()} - {movies['release_date_year'].max()}")
print(f"Title examples:")
for i, title in enumerate(movies['title'].head(3)):
    print(f"  {i+1}. {title}")

PROCESSED MOVIES TABLE
   movieId       title release_date  Action  Adventure  Animation  Children's  \
0        1   toy story   1995-01-01       0          0          1           1   
1        2   goldeneye   1995-01-01       1          1          0           0   
2        3  four rooms   1995-01-01       0          0          0           0   
3        4  get shorty   1995-01-01       1          0          0           0   
4        5     copycat   1995-01-01       0          0          0           0   

   Comedy  Crime  Documentary  ...  Horror  Musical  Mystery  Romance  Sci-Fi  \
0       1      0            0  ...       0        0        0        0       0   
1       0      0            0  ...       0        0        0        0       0   
2       0      0            0  ...       0        0        0        0       0   
3       1      0            0  ...       0        0        0        0       0   
4       0      1            0  ...       0        0        0        0       0   

   

In [20]:
movies.describe().T

,count,mean,min,25%,50%,75%,max,std
movieId,1682.0,841.5,1.0,421.25,841.5,1261.75,1682.0,485.695893
release_date,1681,1989-07-16 12:53:32.373587072,1922-01-01 00:00:00,1993-01-01 00:00:00,1995-01-01 00:00:00,1996-10-18 00:00:00,1998-10-23 00:00:00,NaN
Action,1682.0,0.149227,0.0,0.0,0.0,0.0,1.0,0.356418
Adventure,1682.0,0.080262,0.0,0.0,0.0,0.0,1.0,0.271779
Animation,1682.0,0.02497,0.0,0.0,0.0,0.0,1.0,0.156081
Children's,1682.0,0.072533,0.0,0.0,0.0,0.0,1.0,0.259445
Comedy,1682.0,0.300238,0.0,0.0,0.0,1.0,1.0,0.458498
Crime,1682.0,0.064804,0.0,0.0,0.0,0.0,1.0,0.246253
Documentary,1682.0,0.029727,0.0,0.0,0.0,0.0,1.0,0.169882
Drama,1682.0,0.431034,0.0,0.0,0.0,1.0,1.0,0.495368


In [21]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1682 entries, 0 to 1681
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   movieId             1682 non-null   int64         
 1   title               1682 non-null   object        
 2   release_date        1681 non-null   datetime64[ns]
 3   Action              1682 non-null   int64         
 4   Adventure           1682 non-null   int64         
 5   Animation           1682 non-null   int64         
 6   Children's          1682 non-null   int64         
 7   Comedy              1682 non-null   int64         
 8   Crime               1682 non-null   int64         
 9   Documentary         1682 non-null   int64         
 10  Drama               1682 non-null   int64         
 11  Fantasy             1682 non-null   int64         
 12  Film-Noir           1682 non-null   int64         
 13  Horror              1682 non-null   int64       

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.3. Users** </font>

In [22]:
users.head()

,userId,age,gender,occupation,zip_code
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213


In [23]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 943 entries, 0 to 942
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   userId      943 non-null    int64 
 1   age         943 non-null    int64 
 2   gender      943 non-null    object
 3   occupation  943 non-null    object
 4   zip_code    943 non-null    object
dtypes: int64(2), object(3)
memory usage: 37.0+ KB


In [24]:
#frequency distribution of all categorical columns
for col in users.select_dtypes(include=['object', 'category']).columns:
    print()
    print(f"--- {col} ---")
    display(users[col].value_counts())


--- gender ---


gender
M    670
F    273
Name: count, dtype: int64


--- occupation ---


occupation
student          196
other            105
educator          95
administrator     79
engineer          67
programmer        66
librarian         51
writer            45
executive         32
scientist         31
artist            28
technician        27
marketing         26
entertainment     18
healthcare        16
retired           14
lawyer            12
salesman          12
none               9
homemaker          7
doctor             7
Name: count, dtype: int64


--- zip_code ---


zip_code
55414    9
55105    6
55337    5
10003    5
20009    5
        ..
24060    1
55413    1
19149    1
02176    1
77841    1
Name: count, Length: 795, dtype: int64

In [27]:
valid_occupations = [
    'administrator', 'artist', 'doctor', 'educator', 'engineer', 'entertainment',
    'executive', 'healthcare', 'homemaker', 'lawyer', 'librarian', 'marketing',
    'none', 'other', 'programmer', 'retired', 'salesman', 'scientist', 'student',
    'technician', 'writer'
]

In [28]:
def cleaning_users_data(df, valid_occupations):
    # Clean column names
    df.columns = ['user_id', 'age', 'gender', 'occupation', 'zip_code']
    
    # Store original count for reporting
    original_count = len(df)
    
    # Remove invalid ages (5-100)
    df = df[df['age'].between(5, 100)]
    age_removed = original_count - len(df)
    
    # Remove invalid genders (only M/F)
    df = df[df['gender'].isin(['M', 'F'])]
    gender_removed = original_count - len(df) - age_removed
    
    # Convert gender to dummy variables
    df['gender'] = df['gender'].map({'M': 1, 'F': 0})
    
    # Convert occupation to lowercase for consistency
    df['occupation'] = df['occupation'].str.lower()
    
    # Check for occupation mismatches
    invalid_occupations = df[~df['occupation'].isin(valid_occupations)]
    occupation_removed = len(invalid_occupations)
    
    # Remove invalid occupations
    df = df[df['occupation'].isin(valid_occupations)]
    
    # Store results for reporting
    cleaning_info = {
        'original_count': original_count,
        'age_removed': age_removed,
        'gender_removed': gender_removed,
        'occupation_removed': occupation_removed,
        'invalid_occupations_found': invalid_occupations['occupation'].unique().tolist() if occupation_removed > 0 else []
    }
    
    return df, cleaning_info

users, cleaning_info = cleaning_users_data(users, valid_occupations)

In [29]:
# Display the processing report
print("OCCUPATION VALIDATION REPORT")
print("=" * 50)
print(f"Original users: {cleaning_info['original_count']}")
print(f"Removed due to invalid age: {cleaning_info['age_removed']}")
print(f"Removed due to invalid gender: {cleaning_info['gender_removed']}")
print(f"Removed due to invalid occupation: {cleaning_info['occupation_removed']}")

if cleaning_info['invalid_occupations_found']:
    print(f"Invalid occupations found: {cleaning_info['invalid_occupations_found']}")
else:
    print("✅ All occupations matched perfectly!")

print(f"Final users: {len(users)}")

print("\n" + "=" * 50)
print("PROCESSED USERS TABLE")
print("=" * 50)
print(users.head())

print("\n" + "=" * 50)
print("FINAL SUMMARY")
print("=" * 50)
print(f"Total users: {len(users)}")
print(f"Columns: {list(users.columns)}")
print(f"Age range: {users['age'].min()} - {users['age'].max()}")
print(f"Gender distribution: {users['gender'].value_counts().to_dict()}")
print(f"Unique occupations: {users['occupation'].nunique()}")
print(f"All occupations: {sorted(users['occupation'].unique().tolist())}")

OCCUPATION VALIDATION REPORT
Original users: 943
Removed due to invalid age: 0
Removed due to invalid gender: 0
Removed due to invalid occupation: 0
✅ All occupations matched perfectly!
Final users: 943

PROCESSED USERS TABLE
   user_id  age  gender  occupation zip_code
0        1   24       1  technician    85711
1        2   53       0       other    94043
2        3   23       1      writer    32067
3        4   24       1  technician    43537
4        5   33       0       other    15213

FINAL SUMMARY
Total users: 943
Columns: ['user_id', 'age', 'gender', 'occupation', 'zip_code']
Age range: 7 - 73
Gender distribution: {1: 670, 0: 273}
Unique occupations: 21
All occupations: ['administrator', 'artist', 'doctor', 'educator', 'engineer', 'entertainment', 'executive', 'healthcare', 'homemaker', 'lawyer', 'librarian', 'marketing', 'none', 'other', 'programmer', 'retired', 'salesman', 'scientist', 'student', 'technician', 'writer']


<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.4. Genres** </font>

In [30]:
genres.head()

,genre,genreId
0,unknown,0
1,Action,1
2,Adventure,2
3,Animation,3
4,Children's,4


In [31]:
genres.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   genre    19 non-null     object
 1   genreId  19 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 436.0+ bytes


In [32]:
def cleaning_genres_data(df):
    # Store original info for reporting
    original_count = len(df)
    original_columns = list(df.columns)
    
    # Remove empty columns
    columns_to_remove = []
    for column in df.columns:
        if len(df[column].value_counts()) == 0:
            columns_to_remove.append(column)
    
    # Remove all identified empty columns
    if columns_to_remove:
        df.drop(columns_to_remove, axis=1, inplace=True)
    
    # Remove duplicates
    duplicates_count = df.duplicated().sum()
    df = df.drop_duplicates()
    
    # Store cleaning info for reporting
    cleaning_info = {
        'original_count': original_count,
        'original_columns': original_columns,
        'columns_removed': columns_to_remove,
        'duplicates_removed': duplicates_count,
        'final_count': len(df),
        'final_columns': list(df.columns)
    }
    
    return df, cleaning_info

genres, cleaning_info = cleaning_genres_data(genres)

In [33]:
# Display the processing report
print("GENRES CLEANING REPORT")
print("=" * 50)
print(f"Original entries: {cleaning_info['original_count']}")
print(f"Final entries: {cleaning_info['final_count']}")
print(f"Duplicates removed: {cleaning_info['duplicates_removed']}")

if cleaning_info['columns_removed']:
    print(f"Empty columns removed: {cleaning_info['columns_removed']}")
else:
    print("No empty columns removed")

print(f"Original columns: {cleaning_info['original_columns']}")
print(f"Final columns: {cleaning_info['final_columns']}")

print("\n" + "=" * 50)
print("PROCESSED GENRES TABLE")
print("=" * 50)
print(genres.head())

print("\n" + "=" * 50)
print("FINAL SUMMARY")
print("=" * 50)
print(f"Total genres: {len(genres)}")
print(f"Sample genres: {genres.iloc[:, 0].head(10).tolist()}")

GENRES CLEANING REPORT
Original entries: 19
Final entries: 19
Duplicates removed: 0
No empty columns removed
Original columns: ['genre', 'genreId']
Final columns: ['genre', 'genreId']

PROCESSED GENRES TABLE
        genre  genreId
0     unknown        0
1      Action        1
2   Adventure        2
3   Animation        3
4  Children's        4

FINAL SUMMARY
Total genres: 19
Sample genres: ['unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy']


<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.5. Occupations** </font>

In [34]:
occupations.head()

,occupation
0,administrator
1,artist
2,doctor
3,educator
4,engineer


In [35]:
occupations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   occupation  21 non-null     object
dtypes: object(1)
memory usage: 300.0+ bytes


In [36]:
occupations['occupation'].values

array(['administrator', 'artist', 'doctor', 'educator', 'engineer',
       'entertainment', 'executive', 'healthcare', 'homemaker', 'lawyer',
       'librarian', 'marketing', 'none', 'other', 'programmer', 'retired',
       'salesman', 'scientist', 'student', 'technician', 'writer'],
      dtype=object)

In [37]:
def cleaning_occupations_data(df):
    # Store original info for reporting
    original_count = len(df)
    original_columns = list(df.columns)
    
    # Remove empty columns
    columns_to_remove = []
    for column in df.columns:
        if len(df[column].value_counts()) == 0:
            columns_to_remove.append(column)
    
    # Remove all identified empty columns
    if columns_to_remove:
        df.drop(columns_to_remove, axis=1, inplace=True)
    
    # Remove duplicates
    duplicates_count = df.duplicated().sum()
    df = df.drop_duplicates()
    
    # Store cleaning info for reporting
    cleaning_info = {
        'original_count': original_count,
        'original_columns': original_columns,
        'columns_removed': columns_to_remove,
        'duplicates_removed': duplicates_count,
        'final_count': len(df),
        'final_columns': list(df.columns)
    }
    
    return df, cleaning_info

occupations, cleaning_info = cleaning_occupations_data(occupations)

In [38]:
# Display the processing report
print("OCCUPATIONS CLEANING REPORT")
print("=" * 50)
print(f"Original entries: {cleaning_info['original_count']}")
print(f"Final entries: {cleaning_info['final_count']}")
print(f"Duplicates removed: {cleaning_info['duplicates_removed']}")

if cleaning_info['columns_removed']:
    print(f"Empty columns removed: {cleaning_info['columns_removed']}")
else:
    print("No empty columns removed")

print(f"Original columns: {cleaning_info['original_columns']}")
print(f"Final columns: {cleaning_info['final_columns']}")

print("\n" + "=" * 50)
print("PROCESSED OCCUPATIONS TABLE")
print("=" * 50)
print(occupations.head())

print("\n" + "=" * 50)
print("FINAL SUMMARY")
print("=" * 50)
print(f"Total occupations: {len(occupations)}")
print(f"All occupations: {occupations.iloc[:, 0].tolist()}")

OCCUPATIONS CLEANING REPORT
Original entries: 21
Final entries: 21
Duplicates removed: 0
No empty columns removed
Original columns: ['occupation']
Final columns: ['occupation']

PROCESSED OCCUPATIONS TABLE
      occupation
0  administrator
1         artist
2         doctor
3       educator
4       engineer

FINAL SUMMARY
Total occupations: 21
All occupations: ['administrator', 'artist', 'doctor', 'educator', 'engineer', 'entertainment', 'executive', 'healthcare', 'homemaker', 'lawyer', 'librarian', 'marketing', 'none', 'other', 'programmer', 'retired', 'salesman', 'scientist', 'student', 'technician', 'writer']


<a class="anchor" id='import'>
<font color = '#006400'>
    
# **3. Convert to Parquet** </font>
</a>

In [39]:
# Define paths
output_dir = "../results"

# Make sure the directory exists
import os
os.makedirs(output_dir, exist_ok=True)

# Convert and save
ratings.to_parquet(os.path.join(output_dir, "ratings.parquet"), index=False)
movies.to_parquet(os.path.join(output_dir, "movies.parquet"), index=False)
users.to_parquet(os.path.join(output_dir, "users.parquet"), index=False)
genres.to_parquet(os.path.join(output_dir, "genres.parquet"), index=False)
occupations.to_parquet(os.path.join(output_dir, "occupations.parquet"), index=False)